#  Railroad Blocking Problem  -  MIP with Pyomo & HiGHS

## Why a Mixed-Integer Program?

The RBP is **NP-hard** (it subsumes the Fixed-Charge Network Design problem).  
A **MIP formulation** finds the provably optimal solution with a certified gap,  
at the cost of exponential worst-case runtime.

On this toy instance (8 yards, 50 commodities), the MIP solves to **0% gap in seconds**.

## Mathematical Formulation

### Sets
- $K$  -  commodities (OD pairs with demand $d_k$)
- $B$  -  candidate blocks (yard pairs $(i,j)$ satisfying C4 and C6)
- $Y$  -  yards

### Decision Variables
- $y_{ij} \in \{0,1\}$  -  open block $(i,j)$ ?
- $z_{k,ij} \in \{0,1\}$  -  does commodity $k$ use block $(i,j)$ ?

### Objective
$$\min \sum_{(i,j)\in B} f_{ij} \cdot y_{ij} \;+\; \tau \sum_{k\in K}\sum_{(i,j)\in B} d_k \cdot dist_{ij} \cdot z_{k,ij} \;+\; \sum_{k\in K}\sum_{(i,j)\in B} d_k \cdot h_j \cdot z_{k,ij} \cdot \mathbb{1}[j \neq dest_k]$$

### Key Constraints

| Label | Constraint | Description |
|---|---|---|
| C1 | $\sum_j z_{k,ij} - \sum_j z_{k,ji} = b_k(i)$ | Flow conservation at each yard |
| C2 | $\sum_j y_{ij} \leq T_i$ | Track limit at yard $i$ |
| C3 | $\sum_k d_k \sum_j z_{k,ij} \leq H_i$ | Handling capacity at yard $i$ |
| C4a | $\sum_k d_k z_{k,ij} \geq V_{ij} \cdot y_{ij}$ | Min volume per block |
| C4b | $\sum_k d_k z_{k,ij} \leq M \cdot y_{ij}$ | Linking: use requires opening |

> **Reference**: Magnanti, T.L. & Wong, R.T. (1984). *Network design and transportation planning.*  
> Transportation Science, 18(1), 1-55.  
> Barnhart, C., Jin, H. & Vance, P.H. (2000). *Railroad blocking: A network design application.*  
> Operations Research, 48(4), 603-616.

In [ ]:
import sys
import os
import glob
from pathlib import Path

# -- Environment detection ---------------------------------------------------
ON_KAGGLE = (
    bool(os.environ.get('KAGGLE_DATA_PROXY_TOKEN'))
    or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
    or Path('/kaggle/input').exists()
)

def _find_kaggle_path(slug: str) -> Path:
    """Find dataset/competition by slug anywhere in /kaggle/input."""
    base = Path('/kaggle/input')
    # 1. Old flat layout
    if (base / slug).exists():
        return base / slug
    # 2. New hierarchical layout - scan up to 3 levels
    for pattern in ['/kaggle/input/*/' + slug,
                    '/kaggle/input/*/*/' + slug,
                    '/kaggle/input/*/*/*/' + slug]:
        matches = glob.glob(pattern)
        if matches:
            return Path(sorted(matches)[-1])
    return None

if ON_KAGGLE:
    # Locate competition data and utility dataset
    _comp = _find_kaggle_path('informs-ras-2026-problem-solving-competition')
    _utils = _find_kaggle_path('rbp-railroad-blocking-utils')

    # Debug: show what is available in /kaggle/input
    _tree = sorted(glob.glob('/kaggle/input/**', recursive=False))
    print("Kaggle /input contents:", _tree)

    if _comp is None:
        raise RuntimeError(
            "Competition data not found.\n"
            "Attach competition 'informs-ras-2026-problem-solving-competition' via Add Data."
        )
    if _utils is None:
        _deep = sorted(glob.glob('/kaggle/input/**/**', recursive=False))
        raise RuntimeError(
            "Dataset not found: rbp-railroad-blocking-utils\n"
            "Deep scan: " + str(_deep[:20]) + "\n"
            "Attach the dataset 'nbridelancetb/rbp-railroad-blocking-utils' via Add Data."
        )

    DATA_DIR = _comp
    UTILS_ROOT = _utils
    print("DATA_DIR  :", DATA_DIR)
    print("UTILS_ROOT:", UTILS_ROOT)

    sys.path.insert(0, str(UTILS_ROOT))
    # If src.zip was extracted flat (no src/ subdirectory), build namespace package
    if not (UTILS_ROOT / 'src').exists():
        import types as _t
        _m = _t.ModuleType('src')
        _m.__path__ = [str(UTILS_ROOT)]
        _m.__package__ = 'src'
        sys.modules['src'] = _m
        print("Note: src namespace created at", UTILS_ROOT)
else:
    # Local: search upward for project root (contains src/data_loader.py)
    ROOT = Path().resolve()
    for _ in range(8):
        if (ROOT / 'src' / 'data_loader.py').exists():
            break
        ROOT = ROOT.parent
    else:
        raise RuntimeError(
            "Project root not found from: " + str(Path().resolve()) + "\n"
            "Launch Jupyter/VS Code from the RASPSC2026 folder."
        )
    DATA_DIR = ROOT / 'data'
    sys.path.insert(0, str(ROOT))
    print("Local ROOT:", ROOT)

from src.data_loader import load_rail_network
from src.solvers.greedy import solve_greedy
from src.solvers.mip import solve_mip
from src.validator import validate
from src.evaluator import evaluate
print('Modules loaded OK')
print('  Data dir :', DATA_DIR)

## 1. Load Network & Greedy Warm-Start

In [ ]:
net = load_rail_network(DATA_DIR)
print(f'Network: {len(net.yards_df)} yards, {len(net.demands_df)} commodities')

# Greedy as warm-start for MIP
greedy_sol = solve_greedy(net)
greedy_dict = greedy_sol.to_solution_dict(net)
greedy_cost = evaluate(net, greedy_dict).total_cost
print(f'Greedy baseline: ${greedy_cost:,.0f}  ({len(greedy_sol.blocks)} blocks)')

## 2. MIP Formulation  -  Key Implementation Details

Our MIP formulation uses **Pyomo** (abstract model) + **HiGHS** (open-source MILP solver).  
HiGHS is ~10x faster than CBC on these instances and is freely available via `pip install highspy`.

### Candidate Block Generation

Rather than creating binary variables for all $Y^2$ yard pairs, we filter *admissible* blocks:
- Aggregated OD volume >= C4 minimum volume threshold
- Shortest path distance <  (physically connected)
- Block route <= 1.3 x shortest path (C6 circuitous ratio)

This **reduces the MIP size** significantly.

### Warm-Start Strategy

The greedy solution is passed as an initial feasible solution, which:
1. Gives HiGHS a valid upper bound immediately
2. Prunes the branch-and-bound tree
3. Typically reduces solve time by 50-80%

In [ ]:
import time

print('Solving MIP (Pyomo + HiGHS)...')
t0 = time.time()

mip_sol = solve_mip(
    net,
    warm_start=greedy_sol,         # warm-start from greedy
    time_limit=300.0,              # 5-minute budget
    mip_gap=0.001,                 # stop at 0.1% optimality gap
    verbose=False
)

elapsed = time.time() - t0
mip_dict = mip_sol.to_solution_dict(net)
mip_cost = evaluate(net, mip_dict).total_cost

print(f'MIP solved in {elapsed:.1f}s')
print(f'  Objective   : ${mip_sol.obj_value:>12,.0f}')
print(f'  Lower bound : ${mip_sol.lower_bound:>12,.0f}')
print(f'  MIP gap     : {mip_sol.mip_gap*100:.3f}%')
print(f'  Blocks      : {len(mip_sol.blocks)}')
if mip_sol.unrouted:
    print(f'  WARNING  -  unrouted: {mip_sol.unrouted}')

## 3. Validate Constraints C1-C7

In [ ]:
val = validate(net, mip_dict)
print(f'MIP solution  -  feasibility check:')
print(f'  Feasible (C1-C7): {val.is_feasible}')
if val.violations:
    for v in val.violations:
        print(f'  VIOLATION: {v}')
else:
    print('  All constraints satisfied ')

## 4. Greedy vs MIP Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

greedy_eval = evaluate(net, greedy_dict)
mip_eval    = evaluate(net, mip_dict)

improvement = (greedy_cost - mip_cost) / greedy_cost * 100

# Styled comparison table
data = {
    'Metric': ['Total cost', 'Fixed cost', 'Transport cost', 'Handling cost',
               'Blocks opened', 'MIP gap'],
    'Greedy': [f'${greedy_eval.total_cost:>12,.0f}', f'${greedy_eval.fixed_cost:>12,.0f}',
               f'${greedy_eval.transport_cost:>12,.0f}', f'${greedy_eval.handling_cost:>12,.0f}',
               str(len(greedy_sol.blocks)), 'N/A'],
    'MIP': [f'${mip_eval.total_cost:>12,.0f}', f'${mip_eval.fixed_cost:>12,.0f}',
            f'${mip_eval.transport_cost:>12,.0f}', f'${mip_eval.handling_cost:>12,.0f}',
            str(len(mip_sol.blocks)), f'{mip_sol.mip_gap*100:.3f}%'],
    'Delta': [f'{improvement:+.2f}%', '', '', '', '', ''],
}
df = pd.DataFrame(data)
display(df.style
        .set_caption(f'Greedy vs MIP  -  {improvement:.1f}% cost reduction')
        .set_properties(subset=['MIP'], **{'font-weight': 'bold', 'color': '#155724'})
        .hide(axis='index'))

# Stacked bar with value labels
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#f8f9fa')

x = np.array([0, 1])
cost_components = {
    'Fixed':     [greedy_eval.fixed_cost,     mip_eval.fixed_cost],
    'Transport': [greedy_eval.transport_cost, mip_eval.transport_cost],
    'Handling':  [greedy_eval.handling_cost,  mip_eval.handling_cost],
}
colors_bar = ['#4e79a7', '#f28e2b', '#e15759']
bottom = np.zeros(2)
for (label, vals), color in zip(cost_components.items(), colors_bar):
    bars = ax.bar(x, vals, 0.45, bottom=bottom, label=label, color=color, edgecolor='white')
    # Value labels inside bars (only if large enough)
    for xi, (v, b) in enumerate(zip(vals, bottom)):
        if v > 20000:
            ax.text(xi, b + v / 2, f'${v/1e3:.0f}k', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    bottom += np.array(vals)

# Total labels on top
for xi, total in enumerate([greedy_eval.total_cost, mip_eval.total_cost]):
    ax.text(xi, total + 8000, f'${total/1e6:.3f}M', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['Greedy', f'MIP (-{improvement:.1f}%)'], fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1e6:.2f}M'))
ax.legend(loc='upper right', framealpha=0.9)
ax.set_title('Cost Breakdown: Greedy vs MIP', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


## 5. Block Volume Analysis

In [ ]:
yard_names = net.yards_df.set_index('node_id')['name'].to_dict()

blocks_df = pd.DataFrame(mip_sol.blocks)
blocks_df['from'] = blocks_df['from_yard_id'].map(yard_names)
blocks_df['to']   = blocks_df['to_yard_id'].map(yard_names)
blocks_df = blocks_df[['block_id', 'from', 'to', 'commodity_type', 'block_volume']]
blocks_df = blocks_df.sort_values('block_volume', ascending=False)

print(f'MIP opened {len(blocks_df)} blocks (vs {len(greedy_sol.blocks)} greedy):')
blocks_df

## 6. Generate Submission

In [ ]:
import json

output_path = Path('/kaggle/working/solution_result.json') if ON_KAGGLE else Path('solution_result.json')
csv_path    = Path('/kaggle/working/submission.csv')       if ON_KAGGLE else Path('submission.csv')

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(mip_dict, f, ensure_ascii=False, indent=2)

submission = pd.DataFrame({'ID': [0], 'data': [json.dumps(mip_dict, ensure_ascii=False)]})
submission.to_csv(csv_path, index=False)

print(f'Submission saved: {csv_path}')
print(f'MIP score: ${mip_eval.total_cost:,.0f} (gap={mip_sol.mip_gap*100:.3f}%)')